# EOG historical financial record

**Scope:** organize and analyze the five years of WRDS/Compustat history already retrieved. This notebook does not build forecast assumptions, a forecast, WACC, terminal value, or a DCF.

The report contains FY2021–FY2025. Raw Compustat records remain unchanged in `data/raw/wrds/`. The tables below distinguish provider-reported fields from calculations. Missing items stay blank and are listed for follow-up. The extract's field metadata has no unit-scale descriptions, so values are shown in provider units; confirm the scale before labeling amounts `$ millions` or using them in valuation work.

## 1. Load the project and latest WRDS report

In [ ]:
from pathlib import Path
from decimal import Decimal
import json
import pandas as pd

current = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (current, *current.parents) if (p / "app" / "wrds_client.py").is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find app/wrds_client.py. Start JupyterLab from the DCF-Model project folder.")

report_files = sorted((PROJECT_ROOT / "data" / "raw" / "wrds").glob("*/verification.json"))
if not report_files:
    raise FileNotFoundError("No WRDS report found. See notebooks/README.md to retrieve one first.")
report_path = report_files[-1]
report = json.loads(report_path.read_text(encoding="utf-8"))
compustat = report["compustat"]
if compustat["status"] != "available":
    raise RuntimeError(f"Latest WRDS report status is {compustat['status']}; inspect its attempts before proceeding.")

records = sorted(compustat["records"], key=lambda row: int(row["fyear"]))
years = [int(row["fyear"]) for row in records]
by_year = {int(row["fyear"]): row for row in records}
print(f"Source: {report_path.relative_to(PROJECT_ROOT)}")
print(f"Provider/table: WRDS / {compustat['table']}")
print(f"Retrieved UTC: {report['retrieved_at_utc']}")
print(f"Fiscal years: {years}")

## 2. Build a standardized historical table

The table uses one column per fiscal year. `Provider reported` values are direct fields from the saved Compustat records. `Calculated` values show their formula. `Unavailable` values are intentionally blank because this extract does not contain the required source fields. No missing value is treated as zero.

In [ ]:
def amount(year, field):
    value = by_year[year].get(field)
    return None if value in (None, "") else Decimal(str(value))

def total_debt(year):
    current_debt, long_term_debt = amount(year, "dlc"), amount(year, "dltt")
    return None if current_debt is None or long_term_debt is None else current_debt + long_term_debt

def calculated_ebitda(year):
    ebit, da = amount(year, "oiadp"), amount(year, "dp")
    return None if ebit is None or da is None else ebit + da

history_rows = [
    ("Revenue", "Provider reported", "revt — Revenue - Total", "USD; scale unverified", lambda y: amount(y, "revt")),
    ("Operating expenses", "Unavailable", "Not included in this extract", "USD; scale unverified", lambda y: None),
    ("EBITDA", "Calculated", "oiadp + dp (operating income after depreciation + D&A)", "USD; scale unverified", calculated_ebitda),
    ("D&A / DD&A", "Provider reported", "dp — Depreciation and Amortization", "USD; scale unverified", lambda y: amount(y, "dp")),
    ("EBIT", "Provider reported", "oiadp — Operating Income After Depreciation", "USD; scale unverified", lambda y: amount(y, "oiadp")),
    ("Interest expense", "Unavailable", "Not included in this extract", "USD; scale unverified", lambda y: None),
    ("Pretax income", "Unavailable", "Not included in this extract", "USD; scale unverified", lambda y: None),
    ("Taxes", "Unavailable", "Not included in this extract", "USD; scale unverified", lambda y: None),
    ("Net income", "Provider reported", "ni — Net Income (Loss)", "USD; scale unverified", lambda y: amount(y, "ni")),
    ("Operating cash flow", "Provider reported", "oancf — Operating Activities - Net Cash Flow", "USD; scale unverified", lambda y: amount(y, "oancf")),
    ("Capex", "Provider reported", "capx — Capital Expenditures", "USD; scale unverified", lambda y: amount(y, "capx")),
    ("Change in NWC", "Unavailable", "Required balance-sheet components/change not in this extract", "USD; scale unverified", lambda y: None),
    ("Cash", "Provider reported", "che — Cash and Short-Term Investments", "USD; scale unverified", lambda y: amount(y, "che")),
    ("Total debt", "Calculated", "dlc + dltt (current debt + long-term debt)", "USD; scale unverified", total_debt),
    ("Diluted shares", "Provider reported", "cshfd — Fully diluted shares used for EPS", "Provider share units; scale unverified", lambda y: amount(y, "cshfd")),
    ("Operating income before depreciation", "Provider reported", "oibdp — Operating Income Before Depreciation", "USD; scale unverified", lambda y: amount(y, "oibdp")),
]

history_table = pd.DataFrame([
    {"Metric": metric, "Value type": kind, "Source / formula": source, "Units / scale": units,
     **{str(year): value(year) for year in years}}
    for metric, kind, source, units, value in history_rows
])
display(history_table)

### Read this table carefully

- EBITDA is calculated as reported `oiadp + dp`; it is not a company-adjusted EBITDA measure. Compustat also supplies `oibdp`, shown separately for comparison.
- Total debt is calculated as `dlc + dltt` only when both components are present.
- The extract does not include operating expenses, interest expense, pretax income, taxes, or the balance-sheet history needed for change in NWC. Those cells are blank, not zero.
- The report's field descriptions do not specify scale. Confirm units against the Compustat documentation and filing before assigning `$ millions` or combining with data from another source.

## 3. Calculate historical metrics

Ratios below use the same reported/calculated fields in the table. Revenue growth for the first year is blank because the preceding year is outside this five-year extract. Tax rate and UFCF remain unavailable until the missing inputs are retrieved.

In [ ]:
def ratio(numerator, denominator):
    if numerator is None or denominator in (None, Decimal("0")):
        return None
    return numerator / denominator

metric_specs = [
    ("Revenue growth", "(current-year revenue / prior-year revenue) - 1", "Percent",
     lambda y: None if y == years[0] else (ratio(amount(y, "revt"), amount(years[years.index(y)-1], "revt")) - 1 if ratio(amount(y, "revt"), amount(years[years.index(y)-1], "revt")) is not None else None)),
    ("EBITDA margin", "calculated EBITDA / revenue", "Percent",
     lambda y: ratio(calculated_ebitda(y), amount(y, "revt"))),
    ("EBIT margin", "reported oiadp / revenue", "Percent",
     lambda y: ratio(amount(y, "oiadp"), amount(y, "revt"))),
    ("Capex / revenue", "reported capx / revenue", "Percent",
     lambda y: ratio(amount(y, "capx"), amount(y, "revt"))),
    ("Tax rate", "taxes / pretax income", "Percent — unavailable",
     lambda y: None),
    ("Historical UFCF", "EBIT × (1 - tax rate) + D&A - capex - change in NWC", "USD — unavailable",
     lambda y: None),
]

metric_table = pd.DataFrame([
    {"Metric": name, "Formula": formula, "Units": unit,
     **{str(year): value(year) for year in years}}
    for name, formula, unit, value in metric_specs
])
# Format ratios for readability while leaving unavailable values empty.
for year in map(str, years):
    metric_table[year] = metric_table[year].map(lambda value: None if pd.isna(value) else float(value) * 100)
display(metric_table)

Historical UFCF cannot be calculated faithfully yet: taxes/pretax income and change in NWC are missing. Operating cash flow is reported separately and is not substituted for UFCF.

## 4. E&P operating-history checklist — next data step

These operating metrics are not part of the current Compustat annual extract. Retrieve them before choosing forecast assumptions. Preserve the source, reported unit, and fiscal-year mapping for every series.

In [ ]:
operating_metrics = [
    "Oil production", "Natural gas production", "NGL production", "Total production",
    "Realized oil price", "Realized natural gas price", "Realized NGL price",
    "LOE", "Transportation expense", "Production taxes", "G&A",
    "Capital expenditures", "Proved reserves", "WTI benchmark price",
]
operating_status = pd.DataFrame({
    "Operating metric": operating_metrics,
    "Status": ["Not yet retrieved"] * len(operating_metrics),
    "Source / unit": ["To be recorded with source and reported unit"] * len(operating_metrics),
})
display(operating_status)

## 5. Historical business-driver calculations to do after operating data arrives

Once production, realized prices, and cost data are mapped to the same fiscal years, calculate oil mix, production growth, realized-price differentials, LOE/BOE, G&A/BOE, capex/BOE, and EBITDA/BOE. Do not populate these with assumptions or derive them from mismatched units.

The next work is to improve missing historical statement coverage and add sourced E&P operating history. Forecast assumptions, five-year forecasts, UFCF projections, WACC, terminal value, and DCF are later steps.